# Lab Assignment 1: Rock, Paper, Scissors

## Write and Run Your Own Code to Implement a Rock-Paper-Scissors Model

In [16]:
# Library Declarations
import numpy as np
import random

np.random.seed(0)
random.seed(0)


In [17]:
# Data extraction here: extract the data from the txt file to python input
# One suggestion is using integers 0,1,2 for rock paper scissor
# and using CPU guess, your guess and score (1 if you win; -1 if you lose; 0 for draw) as possible data
demo_data = np.loadtxt(r"./demo_data.txt", dtype=int) .reshape(-1,3)
print("The data in this file has been stored in a NumPy array of size {}.".format(demo_data.shape))
print(demo_data)

The data in this file has been stored in a NumPy array of size (5, 3).
[[ 0  1  1]
 [ 0  2 -1]
 [ 1  0 -1]
 [ 2  0  1]
 [ 0  0  0]]


In [18]:
np.random.seed(0)
random.seed(0)

history = np.loadtxt(r"./training.txt", dtype=int).reshape(-1, 3)

def rock_paper_scissor(history=None, last_choice=None):
    """
    Predict the best move using a 1st-order Markov chain on CPU move history.

    Builds a transition matrix from the CPU's move sequence: transition[i][j]
    counts how often the CPU played move j immediately after move i. Given the
    CPU's most recent move, we predict the most likely next move and return the
    move that beats it.

    Args:
        history (np.ndarray): shape (N, 3) — columns are [cpu_move, player_move, score]
        last_choice: unused (kept for API compatibility)

    Returns:
        int: 0 (Rock), 1 (Paper), or 2 (Scissors)
    """
    # What beats each move: paper(1) beats rock(0), scissors(2) beats paper(1), rock(0) beats scissors(2)
    beats = {0: 1, 1: 2, 2: 0}

    if history is None or len(history) < 2:
        return random.randint(0, 2)

    cpu_moves = history[:, 0]  # column 0 is the CPU's move each round

    # Build 3x3 transition matrix from CPU move history
    transition = np.zeros((3, 3), dtype=int)
    for i in range(len(cpu_moves) - 1):
        transition[cpu_moves[i]][cpu_moves[i + 1]] += 1

    # Predict the CPU's next move based on its last observed move
    last_cpu = cpu_moves[-1]
    row = transition[last_cpu]

    if row.sum() > 0:
        predicted_cpu = int(np.argmax(row))
    else:
        # No prior transitions from this move — fall back to the CPU's most frequent move
        predicted_cpu = int(np.argmax(np.bincount(cpu_moves, minlength=3)))

    return beats[predicted_cpu]


In [19]:
print(f'Algorithm says you should choose: {rock_paper_scissor(history)}')

Algorithm says you should choose: 2


In [20]:
# Model Evaluation
# For each round i > 0, use history[:i] to predict what move to play in round i,
# then check whether that move would have beaten the CPU's actual move in round i.

move_names = {0: "Rock", 1: "Paper", 2: "Scissors"}
wins, losses, ties = 0, 0, 0

for i in range(1, len(history)):
    predicted_player_move = rock_paper_scissor(history[:i])
    cpu_actual = history[i, 0]

    # (predicted - cpu) % 3 == 1  →  predicted beats cpu  →  win
    if (predicted_player_move - cpu_actual) % 3 == 1:
        wins += 1
    elif predicted_player_move == cpu_actual:
        ties += 1
    else:
        losses += 1

total = wins + losses + ties
print("=== Algorithm Evaluation on Training Data ===")
print(f"Rounds evaluated : {total}")
print(f"Wins  : {wins:3d}  ({wins/total*100:.1f}%)")
print(f"Losses: {losses:3d}  ({losses/total*100:.1f}%)")
print(f"Ties  : {ties:3d}  ({ties/total*100:.1f}%)")
decisive = wins + losses
if decisive > 0:
    print(f"\nWin rate (excl. ties): {wins/decisive*100:.1f}%")
    print(f"Random baseline (excl. ties): 50.0%")


=== Algorithm Evaluation on Training Data ===
Rounds evaluated : 99
Wins  :  43  (43.4%)
Losses:  22  (22.2%)
Ties  :  34  (34.3%)

Win rate (excl. ties): 66.2%
Random baseline (excl. ties): 50.0%


## What to Submit?

**Note: This is the first assignment, designed to help you get familiar with the GitHub workflow.**

**For this assignment only**, please submit a `.zip` file containing:
1. Your training data in `training.txt`.
2. Your model evaluation data.
3. A brief write-up describing your algorithm in a couple of sentences, including figures if necessary (your design on paper).
4. Finally, please individually briefly reflect on your experience in a couple of sentences. What are you taking away from this lab?

**Starting from Assignment 2**: Next week, we will release a Google Sheet assigning each student a Lab TA. From Assignment 2 onwards, you will need to:
- Submit a link to your GitHub repository (instead of a .zip file)
- Add your assigned Lab TA as a collaborator to your repository, or they won't be able to grade your work! 